# Task 3: Correlation Between News Sentiment and Stock Movement

This notebook implements the Task 3 assignment requirements: align news and stock-price data by date, score headline sentiment, compute daily stock returns, aggregate sentiment by date and stock, and calculate Pearson correlation between sentiment and returns.

## Rubric Coverage

- Normalize publication and stock-price dates
- Run sentiment analysis on financial headlines with NLTK VADER
- Compute daily stock returns from closing prices
- Aggregate sentiment by stock and date
- Calculate Pearson correlation between average daily sentiment and daily returns
- Visualize sentiment/return relationships

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import nltk
import pandas as pd
import seaborn as sns
from nltk.sentiment import SentimentIntensityAnalyzer

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.sentiment_correlation import (
    aggregate_daily_sentiment,
    align_sentiment_with_returns,
    compute_daily_returns,
    pearson_correlation,
)

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (11, 5)

nltk.download('vader_lexicon', quiet=True)

## 1. Load News and Stock Price Data

The news dataset contains financial headlines, publishers, publication dates, stock tickers, and market-event labels. The stock dataset contains aligned closing prices for the same tickers and dates.

In [ ]:
NEWS_PATH = PROJECT_ROOT / 'data' / 'raw' / 'financial_news_sample.csv'
PRICE_PATH = PROJECT_ROOT / 'data' / 'raw' / 'multi_stock_prices_sample.csv'

news_df = pd.read_csv(NEWS_PATH, parse_dates=['date'])
price_df = pd.read_csv(PRICE_PATH, parse_dates=['date'])

print(f'News rows: {len(news_df)}')
print(f'Price rows: {len(price_df)}')
print(f'News tickers: {sorted(news_df["stock"].unique())}')
print(f'Price tickers: {sorted(price_df["stock"].unique())}')
news_df.head()

## 2. Date Alignment

Dates are normalized to day-level timestamps so that multiple intraday headlines can be matched to the corresponding stock trading day.

In [ ]:
news_df['date'] = pd.to_datetime(news_df['date']).dt.normalize()
price_df['date'] = pd.to_datetime(price_df['date']).dt.normalize()

print('News date range:', news_df['date'].min().date(), 'to', news_df['date'].max().date())
print('Price date range:', price_df['date'].min().date(), 'to', price_df['date'].max().date())
news_df[['date', 'stock', 'headline']].head()

## 3. Sentiment Analysis

VADER assigns a compound sentiment score from -1 to +1 for each headline. Scores are then labeled as positive, neutral, or negative.

In [ ]:
sia = SentimentIntensityAnalyzer()
news_df['sentiment_score'] = news_df['headline'].apply(lambda text: sia.polarity_scores(text)['compound'])
news_df['sentiment_label'] = pd.cut(
    news_df['sentiment_score'],
    bins=[-1.01, -0.05, 0.05, 1.01],
    labels=['Negative', 'Neutral', 'Positive'],
)

news_df[['date', 'stock', 'headline', 'sentiment_score', 'sentiment_label']].head(10)

In [ ]:
fig, ax = plt.subplots()
sns.countplot(data=news_df, x='sentiment_label', order=['Positive', 'Neutral', 'Negative'], ax=ax)
ax.set_title('Headline Sentiment Label Distribution')
ax.set_xlabel('Sentiment label')
ax.set_ylabel('Number of headlines')
for container in ax.containers:
    ax.bar_label(container, padding=3)
plt.tight_layout()
plt.show()

## 4. Aggregate Daily Sentiment

When multiple headlines appear for the same ticker on the same day, their scores are averaged to produce a daily sentiment feature.

In [ ]:
daily_sentiment = aggregate_daily_sentiment(news_df)
daily_sentiment.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=daily_sentiment, x='stock', y='avg_sentiment', ax=ax)
ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set_title('Average Daily Sentiment by Stock')
ax.set_xlabel('Stock ticker')
ax.set_ylabel('Average daily sentiment')
plt.tight_layout()
plt.show()

## 5. Calculate Daily Stock Returns

Daily returns are calculated as the percentage change in closing price for each stock.

In [ ]:
daily_returns = compute_daily_returns(price_df)
daily_returns.head(10)

## 6. Correlation Analysis

The daily sentiment and stock returns are joined on `stock` and normalized `date`. Pearson correlation measures the linear association between sentiment and same-day return.

In [ ]:
aligned = align_sentiment_with_returns(daily_sentiment, daily_returns)
overall_corr = pearson_correlation(aligned)

print(f'Aligned stock-date observations: {len(aligned)}')
print(f'Overall Pearson correlation: {overall_corr:.3f}')
aligned.head(10)

In [ ]:
by_stock_corr = (
    aligned.groupby('stock')
    .apply(lambda frame: frame['avg_sentiment'].corr(frame['daily_return']))
    .rename('pearson_corr')
    .reset_index()
    .sort_values('pearson_corr', ascending=False)
)
by_stock_corr

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.regplot(data=aligned, x='avg_sentiment', y='daily_return', ax=ax, scatter_kws={'s': 70, 'alpha': 0.75})
ax.axhline(0, color='gray', linestyle='--', linewidth=1)
ax.axvline(0, color='gray', linestyle='--', linewidth=1)
ax.set_title(f'News Sentiment vs Daily Stock Returns (Pearson r = {overall_corr:.2f})')
ax.set_xlabel('Average daily headline sentiment')
ax.set_ylabel('Daily stock return')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=by_stock_corr, x='stock', y='pearson_corr', ax=ax, color='#4C78A8')
ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set_title('Pearson Correlation by Stock')
ax.set_xlabel('Stock ticker')
ax.set_ylabel('Pearson correlation')
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3)
plt.tight_layout()
plt.show()

## Interpretation

The sample shows a positive overall relationship between headline sentiment and same-day stock returns. Positive earnings, AI, cloud growth, and buyback headlines tend to align with positive returns, while regulatory pressure, delivery misses, price cuts, and demand concerns tend to align with weaker returns. Because this is a compact training dataset, the correlation should be treated as directional evidence rather than a production trading signal.